# Laboratorio 1 — Series de Tiempo
### Data Science | Universidad del Valle de Guatemala 

**Integrantes:** Adrián González, José Ordoñez, Alejandro Anton

Dataset: ingreso mensual de viajeros internacionales a Guatemala (2009-2026).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
FIGS = "figs"
os.makedirs(FIGS, exist_ok=True)

## Carga de datos

In [ ]:
df = pd.read_excel("Base_Migracion_2009-2026jun.xlsx")
print("Dimensiones:", df.shape)
df.dtypes

## Limpieza y calidad de datos

Revisamos nulos, duplicados, valores atípicos y consistencia de categorías.

In [ ]:
print("--- Nulos por columna ---")
print(df.isna().sum())
print("\n--- Duplicados exactos ---", df.duplicated().sum())

# Fecha de referencia (dia 1 de cada mes)
df["Fecha"] = pd.to_datetime(dict(year=df["Año"], month=df["Mes cod"], day=1))

# Región dos: hay valores '0' y separacion redundante
print("\n--- Valores unicos de 'Región dos' ---")
print(df["Región dos"].value_counts())

In [ ]:
# Unificamos categorias de cruceros dentro de Región dos
df["Región dos"] = df["Región dos"].replace({"Cruceros": "Cruceristas", "0": "Sin Clasificar"})

# Viajero: cantidad de viajeros, se esperan enteros pero hay decimales -> posible
# resultado de agregaciones/estimaciones oficiales. Revisamos negativos y ceros.
print("Registros con Viajero < 0:", (df["Viajero"] < 0).sum())
print("Registros con Viajero == 0:", (df["Viajero"] == 0).sum())
print("Registros con parte decimal no entera:", (df["Viajero"] % 1 != 0).sum())

In [ ]:
# Outliers univariados (a nivel de registro) via IQR
q1, q3 = df["Viajero"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_sup = q3 + 3 * iqr
outliers = df[df["Viajero"] > lim_sup]
print(f"Registros por encima de Q3+3*IQR ({lim_sup:.1f}): {len(outliers)} de {len(df)}")
outliers.groupby("Tipo de Viajero")["Viajero"].agg(["count", "mean", "max"])

## Estadísticas descriptivas

In [ ]:
print("--- Estadísticas descriptivas de Viajero (nivel registro) ---")
print(df["Viajero"].describe())
print("\n--- Total de viajeros por Tipo de Viajero ---")
df.groupby("Tipo de Viajero")["Viajero"].sum().sort_values(ascending=False)

## División Train / Test (70% / 30%, cronológica)

Se ordenan los meses únicos y se corta al 70%, respetando el orden temporal.

In [ ]:
fechas_unicas = sorted(df["Fecha"].unique())
n_fechas = len(fechas_unicas)
corte_idx = int(np.floor(n_fechas * 0.70))
fecha_corte = fechas_unicas[corte_idx - 1]

print(f"Total de meses: {n_fechas}")
print(f"Train (70%): {corte_idx} meses -> {fechas_unicas[0]} a {fecha_corte}")
print(f"Test  (30%): {n_fechas - corte_idx} meses -> {fechas_unicas[corte_idx]} a {fechas_unicas[-1]}")

df_train = df[df["Fecha"] <= fecha_corte].copy()
df_test  = df[df["Fecha"] >  fecha_corte].copy()

print(f"\nRegistros train: {len(df_train):,}")
print(f"Registros test:  {len(df_test):,}")

## Construcción de series de tiempo mensuales

- **Serie obligatoria:** Total mensual de viajeros internacionales.
- **Serie libre** (categoría *Vías de ingreso*): **Vía Aérea**.

In [ ]:
# Serie obligatoria: total mensual de viajeros
serie_total = (
    df_train.groupby("Fecha")["Viajero"].sum().asfreq("MS").sort_index()
)
serie_total.name = "Total_Viajeros"

faltantes_total = serie_total.isna().sum()
print("Serie obligatoria 'Total mensual de viajeros' construida.")
print(f"Meses con hueco tras asfreq('MS'): {faltantes_total}")
if faltantes_total > 0:
    serie_total = serie_total.interpolate(method="linear")
    print("  -> se aplico interpolacion lineal para rellenar huecos.")

In [ ]:
# --- serie libre: Via Aerea ---
serie_aerea = (
    df_train[df_train["Vía"] == "Aérea"]
    .groupby("Fecha")["Viajero"].sum()
    .asfreq("MS").sort_index()
)
serie_aerea.name = "Viajeros_Via_Aerea"

faltantes_aerea = serie_aerea.isna().sum()
print(f"\nSerie libre 'Via Aerea' construida.")
print(f"Meses con hueco tras asfreq('MS'): {faltantes_aerea}")
if faltantes_aerea > 0:
    serie_aerea = serie_aerea.interpolate(method="linear")
    print("  -> se aplico interpolacion lineal para rellenar huecos.")

series_a_analizar = {
    "Total mensual de viajeros (obligatoria)": serie_total,
    "Viajeros via Aerea (serie libre)": serie_aerea,
}